# Ingesting the cust_info into cust_info dataset in bronze layer

In [0]:
%sql
SELECT * FROM CSV.`/Volumes/my_catalog/my_volume/my_file_volume/source_crm/` LIMIT 100


Databricks data profile. Run in Databricks to view.

In [0]:
dbutils.fs.ls("/Volumes/my_catalog/my_volume/my_file_volume/source_crm/")

In [0]:
source_dir = "/Volumes/my_catalog/my_volume/my_file_volume/source_crm/"
schema_dir = "/Volumes/my_catalog/my_volume/my_file_volume/_schemas/cust_info_bronze/"
checkpoint_dir = "/Volumes/my_catalog/my_volume/my_file_volume/_checkpoints/cust_info_bronze/"
target_table = "my_catalog.bronze.cust_info"

df = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .option("cloudFiles.schemaLocation", schema_dir)
      .option("cloudFiles.includeExistingFiles", "true")  
      .option("header", "true")
      .option("pathGlobFilter", "cust_info.csv")            
      .load(source_dir)
)

(df.writeStream
 .format("delta")
 .option("checkpointLocation", checkpoint_dir)
 .option("mergeSchema", "true")  # Allow schema evolution to resolve mismatch
 .outputMode("append")
 .trigger(once=True)                                     
 .toTable(target_table)
)

In [0]:
%sql
SELECT * FROM my_catalog.bronze.cust_info

Databricks data profile. Run in Databricks to view.